<a href="https://colab.research.google.com/github/Manaralmalki1/EnviroGuard/blob/main/AirQuality_Model_Final_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
from sklearn.preprocessing import MinMaxScaler

In [2]:
urban_df = pd.read_csv("UrbanAirPollutionDataset.csv")

In [3]:
urban_df = urban_df.rename(columns={
    "DateTime": "timestamp",
    "PM2.5": "pm25",
    "PM10": "pm10",
    "NO₂": "no2",
    "O₃": "o3",
    "CO": "co",
    "Station_ID": "city"
})

urban_df["timestamp"] = pd.to_datetime(urban_df["timestamp"])

urban_df = urban_df[
    ["timestamp", "city", "pm25", "pm10", "no2", "o3", "co"]
]

In [4]:
beijing_df = pd.read_csv(
    "PRSA_Data_Aotizhongxin_20130301-20170228.csv"
)

In [5]:
beijing_df["timestamp"] = pd.to_datetime(
    beijing_df[["year", "month", "day", "hour"]]
)

beijing_df = beijing_df.rename(columns={
    "PM2.5": "pm25",
    "PM10": "pm10",
    "NO2": "no2",
    "O3": "o3",
    "CO": "co",
    "station": "city"
})

beijing_df = beijing_df[
    ["timestamp", "city", "pm25", "pm10", "no2", "o3", "co"]
]

In [6]:
beijing_df.head(5)

,timestamp,city,pm25,pm10,no2,o3,co
0,2013-03-01 00:00:00,Aotizhongxin,4.0,4.0,7.0,77.0,300.0
1,2013-03-01 01:00:00,Aotizhongxin,8.0,8.0,7.0,77.0,300.0
2,2013-03-01 02:00:00,Aotizhongxin,7.0,7.0,10.0,73.0,300.0
3,2013-03-01 03:00:00,Aotizhongxin,6.0,6.0,11.0,72.0,300.0
4,2013-03-01 04:00:00,Aotizhongxin,3.0,3.0,12.0,72.0,300.0


In [7]:
openaq_df = pd.read_csv("openaq_pl_30.csv")
openaq_df.head(5)

,location,city,country,pollutant,value,timestamp,unit,source_name,latitude,longitude
0,"Solec Kujawski, ul. gen. Roweckiego",Solec Kujawski,PL,pm25,5.9000,2022-05-26 17:00:00+00:00,µg/m³,GIOS,1.0,53.079618
1,"Piotrków Trybunalski, ul. Krakowskie Przedmieście",Piotrków Trybunalski,PL,pm10,16.0000,2022-05-26 17:00:00+00:00,µg/m³,GIOS,1.0,51.404406
2,"Kraków, ul. Bulwarowa",Kraków,PL,co,134.2300,2022-05-26 17:00:00+00:00,µg/m³,GIOS,1.0,50.069308
3,"Gajew, Ujęcie Wody",Gajew,PL,pm10,8.0000,2022-05-26 17:00:00+00:00,µg/m³,GIOS,1.0,52.143250
4,"Szczecin, ul. Andrzejewskiego",Szczecin,PL,pm10,20.1059,2022-05-26 17:00:00+00:00,µg/m³,GIOS,1.0,53.380975


In [8]:
openaq_df["timestamp"] = pd.to_datetime(openaq_df["timestamp"], utc=True)

In [9]:
openaq_pivot = (
    openaq_df
    .pivot_table(
        index=["timestamp", "city", "country", "latitude", "longitude"],
        columns="pollutant",
        values="value",
        aggfunc="mean"
    )
    .reset_index()
)

openaq_pivot.columns.name = None

In [10]:
openaq_pivot.columns = openaq_pivot.columns.str.lower()

In [11]:
openaq_pivot.head(5)

,timestamp,city,country,latitude,longitude,co,no2,o3,pm10,pm25
0,2022-05-19 14:00:00+00:00,Augustów,PL,1.0,53.852639,NaN,NaN,104.00,11.4,4.7
1,2022-05-19 14:00:00+00:00,Biała Podlaska,PL,1.0,52.029194,NaN,NaN,105.51,15.8,NaN
2,2022-05-19 14:00:00+00:00,Białystok,PL,1.0,53.126689,187.0,10.000,NaN,NaN,8.0
3,2022-05-19 14:00:00+00:00,Białystok,PL,1.0,53.129306,NaN,NaN,108.00,NaN,NaN
4,2022-05-19 14:00:00+00:00,Bielsko-Biała,PL,1.0,49.802075,NaN,34.329,NaN,NaN,NaN


In [12]:
common_cols = [
    "timestamp", "city",
    "pm25", "pm10", "no2", "o3", "co"
]

openaq_final = openaq_pivot[common_cols]

In [13]:
for df in [urban_df, beijing_df, openaq_df]:
    df.fillna(df.median(numeric_only=True), inplace=True)

In [14]:
urban_df["timestamp"] = pd.to_datetime(urban_df["timestamp"], errors="coerce")
beijing_df["timestamp"] = pd.to_datetime(beijing_df["timestamp"], errors="coerce")
openaq_df["timestamp"] = pd.to_datetime(openaq_df["timestamp"], utc=True, errors="coerce")

In [15]:
urban_df["timestamp"] = urban_df["timestamp"].dt.tz_localize("UTC")
beijing_df["timestamp"] = beijing_df["timestamp"].dt.tz_localize("UTC")

In [16]:
merged_df = pd.concat(
    [urban_df, beijing_df, openaq_final],
    ignore_index=True
)

In [17]:
import joblib

gas_cols = ["pm25", "pm10", "no2", "o3", "co"]

scaler = MinMaxScaler()
merged_df[gas_cols] = scaler.fit_transform(merged_df[gas_cols])

# حفظ السكالر
joblib.dump(scaler, "air_quality_preprocess_scaler.pkl")
print("Scaler saved successfully")

Scaler saved successfully


In [18]:
cols_to_drop = [
    "location", "country", "pollutant",
    "value", "unit", "source_name",
    "latitude", "longitude"
]

merged_df = merged_df.drop(columns=cols_to_drop, errors='ignore')

In [19]:
merged_df.to_csv(
    "air_quality_preprocessed_v2.csv",
    index=False
)

merged_df.head()

,timestamp,city,pm25,pm10,no2,o3,co
0,2020-01-01 00:00:00+00:00,1,0.117265,0.142591,0.151344,0.117440,0.000073
1,2020-01-01 01:00:00+00:00,1,0.103033,0.141710,0.125523,0.052946,0.000072
2,2020-01-01 02:00:00+00:00,1,0.090033,0.078936,0.132030,0.093869,0.000063
3,2020-01-01 03:00:00+00:00,1,0.082117,0.153568,0.112948,0.061729,0.000077
4,2020-01-01 04:00:00+00:00,1,0.070305,0.133197,0.199402,0.109730,0.000070


In [20]:
print("Number of columns:", merged_df.shape[1])

Number of columns: 7


In [21]:
merged_df.isna().all().sum()

np.int64(0)

In [22]:
final_dataset = pd.read_csv("air_quality_preprocessed_v2.csv")
len(final_dataset)

85271

## Multi-Output Forecasting (1D-CNN)
We forecast **all pollutants** (pm25, pm10, no2, o3, co) for the next 1–3 hours using a 1D-CNN.


In [23]:

import numpy as np
import pandas as pd
from google.colab import drive

from sklearn.preprocessing import MinMaxScaler
import sklearn.metrics as skm

import tensorflow as tf
from tensorflow.keras import layers, models


In [24]:
# load the preprocessed dataset
df = pd.read_csv("air_quality_preprocessed_v2.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df = df.sort_values("timestamp").reset_index(drop=True)

feature_cols = ["pm25", "pm10", "no2", "o3", "co"]

# ensure numeric and handle missing
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median(numeric_only=True))

df.head()


,timestamp,city,pm25,pm10,no2,o3,co
0,2013-03-01 00:00:00+00:00,Aotizhongxin,0.027646,0.036602,0.088195,0.196610,0.030016
1,2013-03-01 01:00:00+00:00,Aotizhongxin,0.031997,0.040534,0.088195,0.196610,0.030016
2,2013-03-01 02:00:00+00:00,Aotizhongxin,0.030909,0.039551,0.097861,0.187322,0.030016
3,2013-03-01 03:00:00+00:00,Aotizhongxin,0.029822,0.038568,0.101083,0.185000,0.030016
4,2013-03-01 04:00:00+00:00,Aotizhongxin,0.026559,0.035619,0.104305,0.185000,0.030016


In [25]:
# تجهيز البيانات
X_all = df[feature_cols].values

# forecast settings
lookback = 24
horizon = 3

In [26]:
def make_windows_multi(X, lookback=24, horizon=1):
    """
    multi-output forecasting:
      input:  (lookback, num_features)
      output: (num_features,) at t + horizon - 1
    """
    X_seq, y_seq = [], []
    for i in range(lookback, len(X) - horizon + 1):
        X_seq.append(X[i-lookback:i, :])
        y_seq.append(X[i + horizon - 1, :])  # all features as targets
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = make_windows_multi(X_all, lookback=lookback, horizon=horizon)
X_seq.shape, y_seq.shape


((85245, 24, 5), (85245, 5))

In [27]:
# time-based split => no shuffle
n = len(X_seq)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X_seq[:train_end], y_seq[:train_end]
X_val,   y_val   = X_seq[train_end:val_end], y_seq[train_end:val_end]
X_test,  y_test  = X_seq[val_end:], y_seq[val_end:]

X_train.shape, X_val.shape, X_test.shape


((59671, 24, 5), (12787, 24, 5), (12787, 24, 5))

In [28]:
# build 1D-CNN multi-output regressor
num_features = X_train.shape[-1]

model = models.Sequential([
    layers.Input(shape=(lookback, num_features)),

    layers.Conv1D(filters=64, kernel_size=3, padding="causal", activation="relu"),
    layers.Conv1D(filters=64, kernel_size=3, padding="causal", activation="relu"),
    layers.MaxPooling1D(pool_size=2),

    layers.Conv1D(filters=128, kernel_size=3, padding="causal", activation="relu"),
    layers.GlobalAveragePooling1D(),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(num_features)  # multi-output: pm25, pm10, no2, o3, co
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 24, 64)         │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 24, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 12, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,661 (182.27 KB)

 Trainable params: 46,661 (182.27 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
# train
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 0.0062 - mae: 0.0520 - val_loss: 0.0014 - val_mae: 0.0281
Epoch 2/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0042 - mae: 0.0422 - val_loss: 0.0013 - val_mae: 0.0278
Epoch 3/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0036 - mae: 0.0386 - val_loss: 0.0014 - val_mae: 0.0278
Epoch 4/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0033 - mae: 0.0369 - val_loss: 0.0013 - val_mae: 0.0278
Epoch 5/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0031 - mae: 0.0359 - val_loss: 0.0014 - val_mae: 0.0286
Epoch 6/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0030 - mae: 0.0353 - val_loss: 0.0014 - val_mae: 0.0292
Epoch 7/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0030 - mae: 0.0350 - val_loss: 0.0015 - val_mae: 0.0279
Epoch 8/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0029 - mae: 0.0348 - val_loss: 0.0013 - val_mae: 0.0280
Epoch 9/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - 

In [30]:
# evaluate => per-feature MAE/RMSE on test

y_pred = model.predict(X_test)

results = []
for j, col in enumerate(feature_cols):
    mae  = skm.mean_absolute_error(y_test[:, j], y_pred[:, j])
    mse  = skm.mean_squared_error(y_test[:, j], y_pred[:, j])
    rmse = np.sqrt(mse)

    results.append((col, mae, rmse))

results_df = pd.DataFrame(results, columns=["feature", "MAE_scaled", "RMSE_scaled"])
results_df

400/400 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


,feature,MAE_scaled,RMSE_scaled
0,pm25,0.018149,0.022727
1,pm10,0.024613,0.030790
2,no2,0.025373,0.031886
3,o3,0.018707,0.023466
4,co,0.003009,0.003085


In [31]:
print(y_test[:5])
print(y_pred[:5])

[[1.16756225e-01 1.37912051e-01 1.57025043e-01 8.30163691e-02
  8.20589479e-05]
 [1.06864453e-01 1.33881123e-01 1.72636404e-01 7.95094572e-02
  7.56771381e-05]
 [9.77913596e-02 1.24738353e-01 1.47471432e-01 9.51235651e-02
  6.22808638e-05]
 [9.80188317e-02 1.25936426e-01 1.45530493e-01 8.67935783e-02
  7.00994041e-05]
 [6.84248973e-02 1.07821609e-01 1.44200438e-01 9.03009774e-02
  6.62767263e-05]]
[[0.08449264 0.11605475 0.14718531 0.08970602 0.00271213]
 [0.08340712 0.11464962 0.14699367 0.08892842 0.002981  ]
 [0.08272615 0.11363795 0.14716156 0.08860587 0.00346947]
 [0.08360396 0.11441991 0.14727734 0.08910891 0.00312758]
 [0.08354734 0.11455236 0.14727661 0.08944187 0.00319374]]


In [32]:
import joblib

scaler_X = joblib.load("air_quality_preprocess_scaler.pkl")

y_pred_raw = scaler_X.inverse_transform(y_pred)
y_test_raw = scaler_X.inverse_transform(y_test)

print("Pred raw:")
print(y_pred_raw[:5])

print("\nTrue raw:")
print(y_test_raw[:5])

Pred raw:
[[56.26551  84.821884 25.308931 30.95924  26.959541]
 [55.267464 83.39253  25.249449 30.624346 29.648266]
 [54.64137  82.363434 25.301558 30.485435 34.533077]
 [55.448444 83.15887  25.337494 30.702078 31.11407 ]
 [55.396385 83.293594 25.337267 30.845476 31.775675]]

True raw:
[[ 85.92925002 107.05584362  28.36292333  28.07817591   0.65838084]
 [ 76.83457125 102.95545141  33.20827606  26.567837     0.59456171]
 [ 68.49260058  93.6551254   25.39773574  33.29244117   0.46059679]
 [ 68.70174267  94.87384501  24.79531998  29.70492503   0.53878346]
 [ 41.49252912  76.44685858  24.38250589  31.21547376   0.50055606]]


In [33]:
df_stats = pd.DataFrame({
    "min": df[feature_cols].min(),
    "max": df[feature_cols].max(),
    "std": df[feature_cols].std(),
})
df_stats

,min,max,std
pm25,0.0,1.0,0.061537
pm10,0.0,1.0,0.068040
no2,0.0,1.0,0.098029
o3,0.0,1.0,0.092607
co,0.0,1.0,0.096940


In [34]:
baseline_pred = X_test[:, -1, :]

rows = []
for j, col in enumerate(feature_cols):
    mae_base = skm.mean_absolute_error(y_test[:, j], baseline_pred[:, j])
    mse_base = skm.mean_squared_error(y_test[:, j], baseline_pred[:, j])
    rmse_base = np.sqrt(mse_base)

    rows.append((col, mae_base, rmse_base))

baseline_df = pd.DataFrame(rows, columns=["feature", "MAE_baseline", "RMSE_baseline"])
baseline_df

,feature,MAE_baseline,RMSE_baseline
0,pm25,0.024566,0.030728
1,pm10,0.033476,0.041961
2,no2,0.035797,0.044996
3,o3,0.026398,0.033022
4,co,0.000022,0.000028


In [35]:
pred_std = np.std(y_pred, axis=0)
true_std = np.std(y_test, axis=0)

pd.DataFrame({
    "feature": feature_cols,
    "std_true": true_std,
    "std_pred": pred_std
})

,feature,std_true,std_pred
0,pm25,0.021834,0.001265
1,pm10,0.029796,0.001788
2,no2,0.031881,0.000427
3,o3,0.023350,0.001029
4,co,0.000020,0.000681


In [36]:
# Save model + scaler for deployment
model.save("cnn_multioutput_air_quality_v2.keras")
import joblib
print("Saved model: cnn_multioutput_air_quality_v2.keras")



Saved model: cnn_multioutput_air_quality_v2.keras


In [37]:
import numpy as np
import joblib
from tensorflow.keras.models import load_model

scaler = joblib.load("air_quality_preprocess_scaler.pkl")
model = load_model("cnn_multioutput_air_quality_v2.keras")

sample_raw = np.array([[60, 100, 30, 40, 1.2]])

print("🔹 Original RAW input:")
print(sample_raw)

sample_scaled = scaler.transform(sample_raw)

print("\n🔹 After scaling (input to model):")
print(sample_scaled)

sample_sequence = np.repeat(sample_scaled[np.newaxis, :, :], 24, axis=1)

print("\n🔹 Shape for model:")
print(sample_sequence.shape)

pred_scaled = model.predict(sample_sequence)

print("\n🔹 Model output (SCALED):")
print(pred_scaled)

pred_raw = scaler.inverse_transform(pred_scaled)

print("\n🔹 Final output (RAW):")
print(pred_raw)

🔹 Original RAW input:
[[ 60.  100.   30.   40.    1.2]]

🔹 After scaling (input to model):
[[8.85544346e-02 1.30975739e-01 1.62299581e-01 1.10698095e-01
  1.36219985e-04]]

🔹 Shape for model:
(1, 24, 5)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 518ms/step

🔹 Model output (SCALED):
[[0.08288399 0.11269538 0.1481961  0.089525   0.00439198]]

🔹 Final output (RAW):
[[54.78649  81.40462  25.622654 30.881279 43.758266]]


In [38]:
model.save("cnn_multioutput_air_quality_v2.h5")